In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

In [ ]:
import os
import torch
from IPython.display import Audio, display
from typing import Optional
from samantha.data.utils import read_audio

from samantha.utils.hdfs_helper import hdfs_ls
from recipes.research.audio_codec.scripts.compile import get_latest_model_from_commit


In [ ]:
# from recipes.research.diff import DiffConfig, DiffInstrumental

# config = DiffConfig(
#     n_layer=4,
#     max_duration=30,
#     sample_rate=44100,
# )
# diff = DiffInstrumental(config)
# diff = diff.to("cuda")
# diff.setup()

In [ ]:
# import torch
# from samantha.data.utils import read_audio

# fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/42 Rey's Theme.mp3"
# # fp = "/mnt/bn/janne-research-xl/assets/music/amy_winehouse/06 - Back To Black.mp3"
# audio, sr = read_audio(fp, diff.config.sample_rate, normalize_loudness=True)
# audio = audio.to("cuda")

# audio = audio[..., : diff.config.sample_rate * diff.config.max_duration]

In [ ]:
# from IPython.display import Audio, display
# from samantha.transforms.audio import batch_plot_spectrogram

# with torch.no_grad():
#     z = diff.get_audio_latents(audio, sr)
#     print(z.shape, z.mean(), z.std())

#     batch_plot_spectrogram(z.cpu().transpose(1, 2), plot_log=False)
#     rec_audio = diff.decode_audio(z)
    
#     print(rec_audio.shape)
#     display(Audio(rec_audio[0].cpu(), rate=diff.config.sample_rate))

## Audio Embedding

In [ ]:
# with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):
#     mulan_emb = diff.get_mulan_embeds(diff.max_seq_len, audio=audio, sample_rate=diff.config.sample_rate)
# print(mulan_emb.shape)

In [ ]:

# seconds_start = [0]
# seconds_total = [30]

# with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):
#     with torch.no_grad():
#         result = diff.get_all_embs(audio, sr, seconds_start, seconds_total)

# Compound Noise Schedule

In [ ]:
# import torch
# from IPython.display import Audio, display

# import numpy as np

# num_samples = 20
# diff.config.noise_scale_factor = 1.0

# x = result.audio_latents
# sigmas = torch.linspace(0, 1, num_samples, device=diff.device)

# noise = torch.randn_like(x)

# alphas, betas = diff.get_alpha_beta(sigmas[:, None, None])
# normalize = False
# for idx in range(num_samples):
#     x_scaled = x / diff.config.noise_scale_factor # TODO mult. or div.
# #     if normalize:
# #         x_scaled = x_scaled / x_scaled.std(dim=(1, 2), keepdims=True)

#     # print("Mean:", x.mean().item(), x_scaled.mean().item())
#     # print("Variance changes more drastically:", x.var().item(), x_scaled.var().item())

#     ## normally:
# #     # noisy_latent = sigmas[idx] * noise + (1 - sigmas[idx]) * x_scaled

#     ## v-objective:
#     noisy_velocity = alphas[idx] * x_scaled + betas[idx] * noise
#     noisy_latent = betas[idx] * noise + alphas[idx] * noisy_velocity # convert back to latent

#     noisy_latent = noisy_latent * diff.config.noise_scale_factor
    
#     print(f"Noise scale: {sigmas[idx].item()}")
#     print("Decoder input:", noisy_latent.mean(), noisy_latent.var(), noisy_latent.shape)
    
#     with torch.no_grad():
#         decoded_audio = diff.decode_audio(noisy_latent.permute(0, 2, 1))
#     display(Audio(decoded_audio[0].cpu(), rate=diff.config.sample_rate))

# DiffInstrumental

In [ ]:
from recipes.research.diff import DiffInstrumental
from recipes.research.audio_codec.scripts.compile import get_latest_model_from_commit

def get_instrumental_diff(commit_hash: str, ckpt_path: Optional[str] = None):
    if ckpt_path is None:
        ckpt_path = get_latest_model_from_commit("diff/default", commit_hash)
        
    diff = DiffInstrumental.load_from_checkpoint(ckpt_path)
    diff = diff.eval().to("cuda")
    print(diff.summarize())
    print(diff.config)
    diff.commit_hash = commit_hash
    diff.commit_step = os.path.basename(ckpt_path)
    return diff


In [ ]:
# diff = get_instrumental_diff("7a291ad")
# diff = get_instrumental_diff("cf1d0a0")
# diff = get_instrumental_diff("a4bb710")
# diff = get_instrumental_diff("7eb9c68") # audio
# diff = get_instrumental_diff("d580348") # clip
# diff = get_instrumental_diff("3cbf2b3") # audio

# diff = get_instrumental_diff("9c4db02")
# diff = get_instrumental_diff("9ffa4a9")
# diff = get_instrumental_diff("ef81b2d", ckpt_path="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/logs/diff/default/ef81b2d/2024-05-14/04-27-05/checkpoints/step=150000.ckpt")

# diff = get_instrumental_diff("ce639f1")

# diff = get_instrumental_diff("adb64d8")


# diff = get_instrumental_diff("fecfa80") # audio, sstk (44.1kHz, 90s)

# diff = get_instrumental_diff("647c632") # mel, step=252000, sstk (44.1kHz, 90s)
# diff = get_instrumental_diff("c4f004e") # mel, bb (24kHz, 90s)

# diff = get_instrumental_diff("20be1e7", ckpt_path="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/logs/diff/default/20be1e7/2024-05-23/10-55-18/checkpoints/step=360000.ckpt") # mel, step=642000, sstk (44.1kHz, 90s)

# diff = get_instrumental_diff("de57d93") # mel, 190s, step=140k
# diff = get_instrumental_diff("2d441dc") # mel, 60s, step=100k

# diff = get_instrumental_diff("4527c17", ckpt_path="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/logs/diff/default/4527c17/2024-05-30/09-47-34/checkpoints/step=300000.ckpt")  # mel, step=251k, 30s (crop_start)

# diff = get_instrumental_diff("7b52555")

# diff = get_instrumental_diff("746b9d0", ckpt_path="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/logs/diff/default/746b9d0/2024-06-04/06-21-31/checkpoints/step=210000.ckpt")

# diff = get_instrumental_diff("92ff525")


# diff = get_instrumental_diff("55ff351")  # v1


# diff = get_instrumental_diff("5c5f7aa")  # 1B
# diff = get_instrumental_diff("ac11aa0")  # 2B

# diff = diff.to("cuda")
# diff.setup()

In [ ]:
# from recipes.research.audio_codec.zoo import AudioCodec_f81b3fa_64l, AudioCodec_7c355ea_64l

# diff = diff.load_audio_codec(AudioCodec_7c355ea_64l())

### DiffCodec (Optional)

In [ ]:
# from typing import Optional
# from recipes.research.diff import DiffCodec
# from recipes.research.audio_codec.scripts.compile import get_latest_model_from_commit

# def get_codec_diff(commit_hash: str, ckpt_path: Optional[str] = None):
#     if ckpt_path is None:
#         ckpt_path = get_latest_model_from_commit("diff/default", commit_hash)
        
#     diff = DiffCodec.load_from_checkpoint(ckpt_path)
#     diff = diff.eval().to("cuda")
#     print(diff.summarize())
#     pprint(diff.config)
#     diff.commit_hash = commit_hash
#     diff.commit_step = os.path.basename(ckpt_path)
#     return diff


In [ ]:
# diff_codec = get_codec_diff("9b6acac") # MelCodec_d100dfd_64l, 250k steps
# diff_codec = get_codec_diff("bdf1e92") # MelCodec_d100dfd_64l_642k, 642k steps
# diff_codec = get_codec_diff("f9c3c29") # MelCodec_f9c3c29_64l_620k (MelTransformer
# diff_codec = get_codec_diff("877367d")
# diff_codec = get_codec_diff("aa3fed3") # MelCodec_6432147_64l_251k=
# diff_codec = get_codec_diff("c1d04d9")
# diff_codec = get_codec_diff("746b9d0")


## Sample

In [ ]:
from recipes.research.diff.prod import load_diffusion_model, sample
from recipes.research.audio_codec.zoo import AudioCodec_f81b3fa_64l, AudioCodec_7c355ea_64l


# commit_hash = "55ff351"  # v1

commit_hash = "5c5f7aa"  # 1B

model = load_diffusion_model(commit_hash, device="cuda:1")
model = model.load_audio_codec(AudioCodec_7c355eda_64l())


In [ ]:
from IPython.display import Audio, display
from samantha.data.av_audio import audio_write

# text = "high quality easy corporate upbeat hopeful health media outro presentation successful advertising learn film success documentary fresh"

text = "high quality 70s disco"

pred_audio, sample_rate = sample(
    model,
    text,
    seconds_start=0,
    seconds_total=60,
    t=50,
    cfg_weight=2.5,
    schedule_tau=1.0,
    solver="dpmpp-3m-sde",
)

audio = audio_write(pred_audio[0].cpu(), sample_rate, format="mp3", mp3_rate=320)
display(Audio(audio, rate=sample_rate))
# display(Audio(pred_audio[0].cpu(), rate=diff.config.sample_rate))

In [ ]:
from samantha.transforms.audio import MelSpectrogram

transform = MelSpectrogram(
    sample_rate=diff.sample_rate,
    n_mels=160,
    n_fft=8192,
).to(diff.device)


mel, _ = transform(pred_audio.mean(dim=1))
batch_plot_spectrogram(mel, plot_log=True, figsize=(20, 10))

In [ ]:
import torchaudio
from tqdm import tqdm

import pandas as pd

prompts_fp = "/mnt/bn/audio-diffusion/peng/sstk_random_30_prompts.csv"
# prompts_fp = "/mnt/bn/audio-diffusion/data/bigmusic_text_prompts/instrumental_prompts_20231030_subset150.csv"
prompts = pd.read_csv(prompts_fp)
text_prompts = prompts["text_prompt"].tolist()

BASE_DIR = os.path.join("/mnt/bn/janne-research-xl/demo", diff.commit_hash, diff.commit_step, os.path.splitext(os.path.basename(prompts_fp))[0])
os.makedirs(BASE_DIR, exist_ok=True)

print("Output dir:", BASE_DIR)

# t = 100
# schedule_tau = 0.3

# for cfg_weight in [1.1, 1.3, 1.5, 1.7, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 4.0]:
for idx, text in enumerate(tqdm(text_prompts)):   
    for t in [200]:
        for cfg_weight in [3.0]:
            for schedule_tau in [0.7]:
                fn =  f"{idx}_{text[:200]}_t={t}_cfg={cfg_weight}_tau={schedule_tau}.flac"
                fn = fn.replace("/", "-")

                fp = os.path.join(BASE_DIR, fn)
                if os.path.exists(fp):
                    continue

                # pred_audio = sample_audio(text, t=t, cfg_weight=cfg_weight, schedule_tau=schedule_tau)
                pred_audio = sample_audio_mel(text, t=t, cfg_weight=cfg_weight, schedule_tau=schedule_tau)
                torchaudio.save(fp, pred_audio[0].cpu(), diff_codec.config.sample_rate)

In [ ]:
# from ipywidgets import interact, interactive, fixed, interact_manual
# from samantha.transforms.audio import batch_plot_spectrogram

# @interact_manual(text_prompt="disco", negative_prompt="", seconds_start=10, seconds_total=180, timesteps=25, cfg_weight=3.0, schedule_tau=0.3)
# def sample(text_prompt: str, negative_prompt: str, seconds_start: int, seconds_total: int, timesteps: int, cfg_weight: float, schedule_tau: float):
#     if negative_prompt == "":
#         negative_prompt = None
    
#     print(f"Inference parameters:\nPrompt: {text_prompt}\nNegative prompt: {negative_prompt}\nTimesteps: {timesteps}\nCFG weight: {cfg_weight}\nTau: {schedule_tau}")
#     batch_text = [text_prompt]
#     batch_seconds_start = [seconds_start]
#     batch_seconds_total = [seconds_total]
#     with torch.no_grad():
#         if negative_prompt is None:
#             batch_negative_prompt = None
#         else:
#             batch_negative_prompt = [negative_prompt]
        
#         pred_z_mel = diff.sample_from_text(batch_text, batch_seconds_start, batch_seconds_total, t=timesteps, cfg_weight=cfg_weight, schedule_tau=schedule_tau, negative_text=batch_negative_prompt)
#         pred_mel = diff.decode_audio(pred_z_mel)
        
        
#         pred_z_audio = diff_codec.sample_z_audio_with_z_mel(pred_z_mel, t=50, cfg_weight=2.0, schedule_tau=0.3)
#         pred_audio = diff_codec.decode_audio(pred_z_audio)
#         print(pred_z_mel.shape, pred_z_audio.shape, pred_audio.shape)
#         batch_plot_spectrogram(pred_mel, plot_log=False)
#     return Audio(pred_audio[0].cpu(), rate=diff.config.sample_rate)

In [ ]:
# from ipywidgets import interact, interactive, fixed, interact_manual

# @interact_manual(text_prompt="disco", negative_prompt="", seconds_start=10, seconds_total=180, timesteps=25, cfg_weight=3.0, schedule_tau=0.3)
# def sample(text_prompt: str, negative_prompt: str, seconds_start: int, seconds_total: int, timesteps: int, cfg_weight: float, schedule_tau: float):
#     if negative_prompt == "":
#         negative_prompt = None
    
#     print(f"Inference parameters:\nPrompt: {text_prompt}\nNegative prompt: {negative_prompt}\nTimesteps: {timesteps}\nCFG weight: {cfg_weight}\nTau: {schedule_tau}")
#     batch_text = [text_prompt]
#     batch_seconds_start = [seconds_start]
#     batch_seconds_total = [seconds_total]
#     with torch.no_grad():
#         if negative_prompt is None:
#             batch_negative_prompt = None
#         else:
#             batch_negative_prompt = [negative_prompt]
#         pred_noise = diff.sample_from_text(batch_text, batch_seconds_start, batch_seconds_total, t=timesteps, cfg_weight=cfg_weight, schedule_tau=schedule_tau, negative_text=batch_negative_prompt)
#         pred_audio = diff.decode_audio(pred_noise)
#     return Audio(pred_audio[0].cpu(), rate=diff.config.sample_rate)

# DiffUMM

In [ ]:
from recipes.research.diff import DiffUMM

def get_umm_diff(commit_hash: str, ckpt_path: Optional[str] = None):
    if ckpt_path is None:
        ckpt_path = get_latest_model_from_commit("diff/default", commit_hash)
        
    diff = DiffUMM.load_from_checkpoint(ckpt_path)
    diff = diff.eval().to("cuda")
    print(diff.summarize())
    pprint(diff.config)
    diff.commit_hash = commit_hash
    diff.commit_step = os.path.basename(ckpt_path)
    return diff


In [ ]:
diff = get_umm_diff("4f69b35")
diff.setup()

In [ ]:
fp = "/mnt/bn/janne-research-xl/assets/music/instrumental/02 - Chevaliers De Sangreal (From The Da Vinci Code Original Motion Picture Soundtrack).mp3"

audio, sr = read_audio(
    fp,
    diff.config.sample_rate,
    normalize_loudness=True
)
audio = audio.to("cuda")

In [ ]:
with torch.no_grad():
    pred_noise = diff.ddim_sample(audio, sr, t=20, cfg_weight=1.0, schedule_tau=0.4)
    pred_audio = diff.decode_audio(pred_noise.permute(0, 2, 1))

In [ ]:
display(Audio(pred_audio[0].cpu(), rate=diff.config.sample_rate))

# Codec2Codec

In [ ]:
import os
import torch
from pprint import pprint
from recipes.research.diff import DiffCodec
from typing import Optional


def get_codec_diff(commit_hash: str, ckpt_path: Optional[str] = None):
    if ckpt_path is None:
        ckpt_path = get_latest_model_from_commit("diff/default", commit_hash)
        
    diff = DiffCodec.load_from_checkpoint(ckpt_path)
    diff = diff.eval().to("cuda")
    print(diff.summarize())
    pprint(diff.config)
    diff.commit_hash = commit_hash
    diff.commit_step = os.path.basename(ckpt_path)
    return diff

In [ ]:
# diff = get_codec_diff("7155a2b")
# diff_codec = get_codec_diff("877367d")

# diff_codec = get_codec_diff("bdf1e92") # MelCodec_d100dfd_64l_642k, 642k steps

diff_codec = get_codec_diff("c1d04d9") # 10s, 10hz


In [ ]:
# import torch
# from samantha.data.utils import read_audio
# from IPython.display import Audio, display

# fp = "/mnt/bn/janne-research-xl/assets/music/bob dylan/03 - Masters of War.mp3"
# audio, sr = read_audio(
#     fp,
#     diff_codec.config.sample_rate,
#     normalize_loudness=True
# )
# audio = audio.to("cuda")

# audio = audio[..., :sr * 30]


audio = batch.audio[1:2].to("cuda")
sr = sample_rate

In [ ]:
with torch.no_grad():
    # z_mel = diff_codec.mel_codec.get_z(audio, sr)
    # rec_mel = diff_codec.mel_codec.decode_z(z_mel)
    
    rec_mel = diff_codec.mel_codec.get_rec_mel(audio, sr)
        
    rec_mel = rec_mel.permute(0, 2, 1)
    z_audio = diff_codec.sample_z_audio_with_z_mel(rec_mel, t=100, cfg_weight=2.0, schedule_tau=0.3)
    pred_audio = diff_codec.decode_audio(z_audio)

In [ ]:
display(Audio(audio.cpu()[0, :, :sr*60], rate=sample_rate))
display(Audio(pred_audio.cpu()[0, :, :sr*60], rate=diff_codec.config.sample_rate))

In [ ]:
import os
import torch
from pprint import pprint
from recipes.research.diff import DiffInstrumental, DiffLyrics
from typing import Optional

def get_diff(commit_hash: str, ckpt_path: Optional[str] = None):
    if ckpt_path is None:
        ckpt_path = get_latest_model_from_commit("diff/default", commit_hash)
        
    diff = DiffLyrics.load_from_checkpoint(ckpt_path)
    diff = diff.eval().to("cuda")
    print(diff.summarize())
    pprint(diff.config)
    diff.commit_hash = commit_hash
    diff.commit_step = os.path.basename(ckpt_path)
    return diff



# diffinstrumental
# diff = get_diff("9eba380", ckpt_path="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/logs/diff/default/9eba380/2024-04-25/15-34-45/checkpoints/step=040000.ckpt")

# difflyrics
# diff = get_diff("9eba380", ckpt_path="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/logs/diff/default/9eba380/2024-04-25/14-58-56/checkpoints/step=110000.ckpt")

diff = get_diff("0712d33")


In [ ]:
diff.setup()

In [ ]:
import torch
from typing import List
from samantha.data.utils import read_audio

def sample(audio, sample_rate: int, lyrics: List[List[str]], artist_names: List[str], t: int, cfg_weight: float, schedule_tau: float):
    audio = audio.to("cuda")
    
    with torch.no_grad():
        pred_noise = diff.ddim_sample(audio, sample_rate, lyrics, artist_names, t, cfg_weight, schedule_tau)
    
    pred_audio = diff.decode_audio(pred_noise.permute(0, 2, 1))
    return pred_audio, audio

In [ ]:
from samantha.data.audio.webdataset import AudioWebDataset


billboard = AudioWebDataset(
    url2index="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/data/music/shards/billboard_hot_200-v2_normalised_-16LUFS/*/url2index.txt",
    sample_rate=diff.config.sample_rate,
    channels=2,
    pad=True,
    segment_duration=diff.config.max_duration,
    min_audio_duration=15,
    max_audio_duration=diff.config.max_duration,
    resampled=True,
    shardshuffle=True,
    shuffle_buffer_size=10,
    lyrics_data_path="hdfs://harunava/home/byte_data_seed_us/hdd_va/speech/data/js/data/music/billboard_hot_200-v2_lyrics/billboard_hot_200-v2_lyrics.p",
)   

In [ ]:
batch = next(iter(billboard))
lyrics = [batch.segment_info.lyrics.text]
artist_names = [batch.index["spotify_primary_artist_name"]]
audio = batch.audio[None]
print(artist_names, lyrics)

In [ ]:
import os
import torchaudio
from IPython.display import Audio, display

fps = [
    "/mnt/bn/janne-research-xl/data/instrumental_hq/TV & Movies/Pirates Of The Caribbean - Soundtrack Treasures Collection/Pirates of the Caribbean - The Curse Of The Black Pearl - Original Soundtrack/15 - He's A Pirate.mp3",
    # "/mnt/bn/janne-research-xl/assets/music/instrumental/42 Rey's Theme.mp3",
    # "/mnt/bn/janne-research-xl/assets/music/instrumental/02 - Chevaliers De Sangreal (From The Da Vinci Code Original Motion Picture Soundtrack).mp3",
    # "/mnt/bn/janne-research-xl/data/instrumental_hq/__Soundtracks(Movie, Anime)/Music from the Harry Potter Films/15 Hedwig's Theme.mp3",
    # "/mnt/bn/janne-research-xl/data/instrumental_hq/__Soundtracks(Movie, Anime)/Music from the Harry Potter Films/1 Harry's Wondrous World.mp3",
    # "/mnt/bn/janne-research-xl/data/instrumental_hq/__Soundtracks(Movie, Anime)/Hans Zimmer/Inception (OST)/03. Dream Is Collapsing.mp3",
    # "/mnt/bn/janne-research-xl/data/instrumental_hq/Cinéma/James Bond/Best Of Bond... James Bond 50 Years - 50 Tracks/Disc 1/1-01 James Bond Theme (From Dr. No.).mp3",
]

t = 50
cfg_weight = 3.0
schedule_tau = 0.3

for fp in fps:
    # audio, sample_rate = read_audio(fp, diff.config.sample_rate, normalize_loudness=True)
    pred_audio, audio = sample(audio, diff.config.sample_rate, lyrics, artist_names, t=t, cfg_weight=cfg_weight, schedule_tau=schedule_tau)
    fn = os.path.basename(fp)
    
    display(Audio(pred_audio[0].cpu(), rate=diff.config.sample_rate))
    break
    # torchaudio.save(os.path.join("/mnt/bn/janne-research-xl/logs/diff/default", diff.commit_hash, diff.commit_step, f"{fn}_real.flac"), audio[0].cpu(), diff.config.sample_rate)
    # torchaudio.save(os.path.join("/mnt/bn/janne-research-xl/demo", diff.commit_hash, diff.commit_step, f"{fn}_t={t}_cfg={cfg_weight}_tau={schedule_tau}_gen.flac"), pred_audio[0].cpu(), diff.config.sample_rate)